In [ ]:
!pip install -q transformers accelerate
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters

import warnings
import transformers
import torch
from datetime import datetime

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from langchain.tools import tool
from langchain_huggingface import HuggingFacePipeline
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore


warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()


# ------------------ Настройка модели ------------------

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tok = AutoTokenizer.from_pretrained(MODEL_ID)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

generator = pipeline(
    task="text-generation",
    model=base_model,
    tokenizer=tok,
    max_new_tokens=1024,
    max_length=None,
    do_sample=False,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=generator)


# ------------------ Инструменты ------------------

@tool
def get_today_date(dummy: str = "") -> str:
    """Возвращает текущую дату."""
    return datetime.now().strftime("%Y-%m-%d")


@tool
def search_knowledge_base(question: str) -> str:
    """Ищет информацию в документах."""
    found_docs = vector_storage.similarity_search(question, k=2)
    return "\n\n".join(doc.page_content for doc in found_docs)


# ------------------ Загрузка текста ------------------

SOURCE_FILE = "Article.txt"

loader = TextLoader(SOURCE_FILE, encoding="cp1251")
loaded_docs = loader.load()

splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=30)
text_parts = splitter.split_documents(loaded_docs)

device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': device}
)

vector_storage = InMemoryVectorStore.from_documents(
    documents=text_parts,
    embedding=embedding_model
)


# ------------------ Агент с Chat-шаблонами ------------------

def analyze_question(question: str) -> str:
    prompt = f"""<|system|>
Choose one tool: 'get_today_date', 'search_knowledge_base', or 'none'. Reply ONLY with the tool name.</s>
<|user|>
Question: {question}</s>
<|assistant|>
"""
    response = llm.invoke(prompt).strip().lower()
    first_line = response.split('\n')[0].strip()

    if 'date' in first_line or 'today' in first_line or 'дата' in question.lower():
        return 'get_today_date'
    elif 'search' in first_line or 'knowledge' in first_line:
        return 'search_knowledge_base'
    else:
        return 'none'


def run_agent(question: str) -> str:
    decision = analyze_question(question)

    if decision == 'get_today_date':
        result = get_today_date.invoke({})
        return f"Сегодняшняя дата: {result}"

    elif decision == 'search_knowledge_base':
        context = search_knowledge_base.invoke({"question": question})
        answer_prompt = f"""<|system|>
Ты ИИ-ассистент. Ответь на вопрос пользователя на основе предоставленного текста. Отвечай кратко на русском языке. Использовать можно только представленный тест</s>
<|user|>
Текст: {context}

Вопрос: {question}</s>
<|assistant|>
"""
        answer = llm.invoke(answer_prompt).strip()
        return answer.split('</s>')[0].strip()

    else:
        answer_prompt = f"""<|system|>
Отвечай на вопросы кратко на русском языке.</s>
<|user|>
Вопрос: {question}</s>
<|assistant|>
"""
        answer = llm.invoke(answer_prompt).strip()
        return answer.split('</s>')[0].strip()


# ------------------ Запуск ------------------

print(run_agent("Какая западная компания известна своими ИИ технологиями?"))
print(run_agent("Какая сегодня дата?"))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.0 which is incompatible.


/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Решение: Google. Google является одним из крупнейших и наиболее известных западных компаний, специализирующихся на разработке и использовании искусственного интеллекта (AI).
Сегодняшняя дата: 2026-05-13
